<a href="https://colab.research.google.com/github/trinhtattran/RAGassistant/blob/main/PD_RAG_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DATA_DIR = "/content/drive/MyDrive/RAG/data"

In [3]:
!pip -q install langchain langchain-community sentence-transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.6/330.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [4]:
!pip install -q pypdf==3.17.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.0 MB/s eta 0:00:00


In [5]:
!pip install -q pdfplumber


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 46.0 MB/s eta 0:00:00


In [6]:
import langchain
import sentence_transformers
import faiss
import pypdf

print("All imports OK")


All imports OK


/usr/local/lib/python3.12/dist-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from this module in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


In [7]:
import os, re
from langchain_community.document_loaders import PDFPlumberLoader #more tables on my docs now

DATA_DIR = "/content/drive/MyDrive/RAG/data"

def load_pdfs(folder_path):
    docs = []
    for fn in os.listdir(folder_path):
        if fn.lower().endswith(".pdf"):
            loader = PDFPlumberLoader(os.path.join(folder_path, fn))
            loaded = loader.load()
            for d in loaded:
                d.metadata["source"] = fn
            docs.extend(loaded)
    return docs

def basic_clean(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    return text.strip()

raw_docs = load_pdfs(DATA_DIR)

for d in raw_docs:
    d.page_content = basic_clean(d.page_content)

print("Loaded pages:", len(raw_docs))
print("Example source:", raw_docs[0].metadata)
print("Example text:", raw_docs[0].page_content[:400])


Loaded pages: 615
Example source: {'source': 'Parkinson’s Disease Pathogenesis and Clinical Aspects.pdf', 'file_path': '/content/drive/MyDrive/RAG/data/Parkinson’s Disease Pathogenesis and Clinical Aspects.pdf', 'page': 0, 'total_pages': 194, 'CreationDate': "D:20181224134218+05'30'", 'Creator': 'Adobe InDesign CS5.5 (7.5)', 'ModDate': "D:20181224134252+05'30'", 'Producer': 'Adobe PDF Library 9.9', 'Trapped': 'False'}
Example text: Parkinson’s Disease Pathogenesis and Clinical Aspects Cover image: A case of Parkinson’s disease as described and illustrated by William Gowers. See page 112, Chapter 6 for details. CP-005.indb 1 24/12/18 1:42 PM


In [8]:
print("Loaded pages:", len(raw_docs))
print("Sample text length:", len(raw_docs[0].page_content))
print(raw_docs[0].page_content[:500])
#testing to see if docs are read

Loaded pages: 615
Sample text length: 212
Parkinson’s Disease Pathogenesis and Clinical Aspects Cover image: A case of Parkinson’s disease as described and illustrated by William Gowers. See page 112, Chapter 6 for details. CP-005.indb 1 24/12/18 1:42 PM


In [9]:
from collections import defaultdict
import numpy as np

# raw_docs is a list of LangChain Documents (one per page)
lengths = [len(d.page_content.strip()) for d in raw_docs]

print("Total pages:", len(raw_docs))
print("Pages with 0 chars:", sum(l == 0 for l in lengths))
print("Pages with <50 chars:", sum(l < 50 for l in lengths))
print("Median chars/page:", int(np.median(lengths)))
print("10th percentile chars/page:", int(np.percentile(lengths, 10)))

# Group by PDF filename
by_pdf = defaultdict(list)
for d in raw_docs:
    by_pdf[d.metadata.get("source","UNKNOWN")].append(len(d.page_content.strip()))

print("\nWorst PDFs by median extracted chars/page:")
stats = []
for pdf, lens in by_pdf.items():
    stats.append((pdf, int(np.median(lens)), sum(l == 0 for l in lens), len(lens)))
stats.sort(key=lambda x: x[1])  # sort by median text length

for pdf, med, zeros, total in stats[:10]:
    print(f"{pdf:45}  median={med:4d}  zero_pages={zeros:3d}/{total}")


Total pages: 615
Pages with 0 chars: 23
Pages with <50 chars: 31
Median chars/page: 3075
10th percentile chars/page: 675

Worst PDFs by median extracted chars/page:
Parkinsonism vs Parkinson’s disease.pdf        median= 999  zero_pages=  0/8
Atypical Parkinsonism BCM.pdf                  median=1308  zero_pages=  0/8
Atypical Parkinsonism.pdf                      median=1435  zero_pages=  0/11
Clinical effectiveness and cost-effectiveness of physiotherapy and occupational therapy versus no therapy in mild to moderate Parkinson’s disease.pdf  median=1631  zero_pages= 23/124
UPDRS.pdf                                      median=1745  zero_pages=  0/8
Atypical Parkinsonian Disorders.pdf            median=2202  zero_pages=  0/4
MDS-UPDRS.pdf                                  median=2408  zero_pages=  0/33
How to approach a patient with parkinsonism - red flags for atypical parkinsonism.pdf  median=2432  zero_pages=  0/34
Stages in Parkinson’s Disease.pdf              median=2829  zero_pages

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)

chunks = splitter.split_documents(raw_docs)

print("Total chunks:", len(chunks))
print("One chunk metadata:", chunks[0].metadata)
print("One chunk text:", chunks[0].page_content[:300]) #need to print out to see


Total chunks: 3054
One chunk metadata: {'source': 'Parkinson’s Disease Pathogenesis and Clinical Aspects.pdf', 'file_path': '/content/drive/MyDrive/RAG/data/Parkinson’s Disease Pathogenesis and Clinical Aspects.pdf', 'page': 0, 'total_pages': 194, 'CreationDate': "D:20181224134218+05'30'", 'Creator': 'Adobe InDesign CS5.5 (7.5)', 'ModDate': "D:20181224134252+05'30'", 'Producer': 'Adobe PDF Library 9.9', 'Trapped': 'False'}
One chunk text: Parkinson’s Disease Pathogenesis and Clinical Aspects Cover image: A case of Parkinson’s disease as described and illustrated by William Gowers. See page 112, Chapter 6 for details. CP-005.indb 1 24/12/18 1:42 PM


In [11]:
import random

def print_random_chunks(chunks, n=5, max_chars=900):
    for idx in random.sample(range(len(chunks)), n):
        c = chunks[idx]
        print("\n" + "="*90)
        print(f"CHUNK #{idx}")
        print("SOURCE:", c.metadata.get("source"), "| PAGE:", c.metadata.get("page"))
        print(c.page_content[:max_chars])

print_random_chunks(chunks, n=5)
#this size works!


CHUNK #331
SOURCE: Parkinson’s Disease Pathogenesis and Clinical Aspects.pdf | PAGE: 75
76. Gegg ME, Burke D, Heales SJ, Cooper JM, Hardy J, Wood NW, et al. Glucocerebrosidase deficiency in substantia nigra of Parkinson disease brains. Ann Neurol. 2012;72(3):455–63. http://dx.doi. org/10.1002/ana.23614 77. Brockmann K, Srulijes K, Pflederer S, Hauser AK, Schulte C, Maetzler W, et al. GBA-associated Parkinson’s disease: Reduced survival and more rapid progression in a prospective longitudinal study. Mov Disord. 2015;30(3):407–11. http://dx.doi.org/10.1002/mds.26071 78. Aharon-Peretz J, Rosenbaum H, Gershoni-Baruch R. Mutations in the glucocerebrosidase gene and Parkinson’s disease in Ashkenazi Jews. N Engl J Med. 2004;351(19):1972–7. http://dx.doi. org/10.1056/NEJMoa033277 79. Tan EK, Tong J, Fook-Chong S, Yih Y, Wong MC, Pavanni R, et al. Glucocerebrosidase mutations and

CHUNK #98
SOURCE: Parkinson’s Disease Pathogenesis and Clinical Aspects.pdf | PAGE: 30
treatments in the future. C

In [12]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# shared cohort model (recommended in spec)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# build the vector store (embeds all chunks once)
db = FAISS.from_documents(chunks, embeddings)

print("FAISS index built.")
print("Total chunks indexed:", len(chunks))


/tmp/ipython-input-1004942619.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.war

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index built.
Total chunks indexed: 3054


In [17]:
def retrieve_top_k(query: str, k: int = 6):
    hits = db.similarity_search(query, k=k)  # returns LangChain Documents
    # each hit has .page_content and .metadata (source/page)
    return hits

def show_hits(hits, max_chars=450):
    for i, h in enumerate(hits, 1):
        print("\n" + "-"*90)
        print(f"HIT {i} | source={h.metadata.get('source')} | page={h.metadata.get('page')}")
        print(h.page_content[:max_chars])


In [18]:
q = "What are red flags for atypical parkinsonism?"
hits = retrieve_top_k(q, k=6)
show_hits(hits)



------------------------------------------------------------------------------------------
HIT 1 | source=Parkinson’s Disease Pathogenesis and Clinical Aspects.pdf | page=142
Aspects. Stoker TB, Greenland JC (Editors). Codon Publications, Brisbane, Australia. ISBN: 978-0-9944381-6-4; Doi: http://dx.doi. org/10.15586/codonpublications.parkinsonsdisease.2018 Copyright: The Authors. Licence: This open access article is licenced under Creative Commons Attribution 4.0 International (CC BY 4.0). https://creativecommons.org/licenses/by-nc/4.0/ 129 CP-005.indb 129 24/12/18 1:42 PM

------------------------------------------------------------------------------------------
HIT 2 | source=Red flags phenotyping.pdf | page=0
knowledge and expertise to capture and interpret. Red flags involve (MSA), or at least hinted at an alternative diagnosis to Parkinson's different body parts and diverse aspects of the nervous system, often disease(PD) [1]. Sincethen, theterm‘red flag’ hasbeenconsistently non-

In [19]:
FAISS_DIR = "/content/drive/MyDrive/RAG/faiss_index"
db.save_local(FAISS_DIR)
print("Saved FAISS index to:", FAISS_DIR)

# Later, reload like this:
# db = FAISS.load_local(FAISS_DIR, embeddings, allow_dangerous_deserialization=True)


Saved FAISS index to: /content/drive/MyDrive/RAG/faiss_index


In [20]:
test_questions = [
  "What are red flags that suggest atypical parkinsonism rather than idiopathic Parkinson's disease?",
  "How does essential tremor differ from Parkinson's disease tremor clinically?",
  "What clinical features distinguish progressive supranuclear palsy (PSP) from Parkinson's disease?",
  "What clinical features distinguish multiple system atrophy (MSA) from Parkinson's disease?",
  "What is drug-induced parkinsonism and how does it present compared to Parkinson's disease?",
  "What are common causes of secondary parkinsonism?",
  "What does bradykinesia mean clinically and how is it assessed?",
  "What does rigidity mean clinically and how is it assessed?",
  "What does the MDS-UPDRS Part III measure?",
  "What is Hoehn and Yahr staging used for?",
  "What non-motor symptoms are commonly associated with Parkinson's disease?",
  "When is dopamine transporter imaging (DaTscan) considered in evaluation?",
  "What are limitations of DaTscan/dopamine transporter imaging?",
  "What features suggest corticobasal syndrome rather than Parkinson's disease?",
  "What features suggest vascular parkinsonism rather than Parkinson's disease?",
  "What is the typical progression pattern of idiopathic Parkinson's disease?",
  "What gait abnormalities are described in Parkinson's disease?",
  "What does postural instability imply in parkinsonism evaluation?",
  "What early autonomic symptoms may suggest atypical parkinsonism?",
  "What does poor response to levodopa suggest about the diagnosis?"
]
print("Total questions:", len(test_questions))


Total questions: 20


In [21]:
for q in test_questions[:10]:
    print("\n" + "="*100)
    print("Q:", q)
    hits = retrieve_top_k(q, k=6)
    # FAISS returns Documents
    for i, h in enumerate(hits[:3], 1):
        print(f"TOP {i}: {h.metadata.get('source')} p{h.metadata.get('page')}")
        print(h.page_content[:220].replace("\n"," "), "...")



Q: What are red flags that suggest atypical parkinsonism rather than idiopathic Parkinson's disease?
TOP 1: How to approach a patient with parkinsonism - red flags for atypical parkinsonism.pdf p28
the accuracy of clinical diagnosis in Parkinson’s disease: A clinicopathologic study. Neurology,42(6),1142. Hughes,A.J.,Colosimo,C.,Kleedorfer,B.,Daniel,S.E.,&Lees,A.J.(1992).Thedopa- minergicresponseinmultiplesystematro ...
TOP 2: Recognizing Atypical Parkinsonisms.pdf p1
reliable biomarkers have been established as diagnostic for any of the atypical parkinsonisms; however, ancillary brain imaging (as described in subsequent sections) is increasingly used as adjunctive diagnostic tools in ...
TOP 3: Parkinson’s Disease Pathogenesis and Clinical Aspects.pdf p142
Aspects. Stoker TB, Greenland JC (Editors). Codon Publications, Brisbane, Australia. ISBN: 978-0-9944381-6-4; Doi: http://dx.doi. org/10.15586/codonpublications.parkinsonsdisease.2018 Copyright: The Authors. Licence: Thi ...

Q: How